# SEIS 606: Vibe Coding
## Homework 2, Image Generation for App Specs
Dante Razo, razo3843@stthomas.edu, FA26

I've been using this GPU-accelerated notebook template since I first started at UST. It's something that I carry from class to class.

## GPU-Accelerated Environment Configuration

In [21]:
import torch

# validate CUDA setup
print("Torch CUDA Available? ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Torch CUDA Version:", torch.version.cuda)
    print("Torch cuDNN Version:", torch.backends.cudnn.version())

    # print GPU information
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:", torch.cuda.get_device_name(device=i))

    # check NVIDIA driver
    !echo && nvidia-smi

# set device type
device: str = "cuda" if torch.cuda.is_available() else "cpu"

Torch CUDA Available?  True
Torch CUDA Version: 13.0
Torch cuDNN Version: 92400

GPU 0: NVIDIA GeForce RTX 5090

Fri Sep 25 00:05:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 615.71.08              KMD Version: 616.92        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0A:00.0  On |                  N/A |
|  0%   44C    P0             65W /  460W |    4177MiB /  32607MiB |      6%      Default |
|                          

In [22]:
import gc


def free_vram() -> None:
    gc.collect()
    torch.cuda.empty_cache()


# free now, though it should be empty with a fresh kernel
free_vram()

In [23]:
import os
from pathlib import Path

# create cache location
hf_home: Path = Path("/cache/huggingface")
hf_home.mkdir(parents=True, exist_ok=True)

# set environment variables for huggingface / transformers
os.environ["HF_HOME"] = str(object=hf_home)

In [24]:
# validate environment variables with assertions
assert hf_home.exists()
assert os.environ["HF_HOME"] == str(object=hf_home)

In [25]:
from dotenv import load_dotenv

# load environment, including HF token
load_dotenv()

False

## Loading the Image Generation Model
I had AI generate a list of models to try given my hardware, and I created the following struct to easily switch between them.

In [ ]:
# define model structs for easy switching
from torch import dtype, float16
from dataclasses import dataclass
from enum import Enum


@dataclass
class Text2ImageModel:
    name: str
    torch_dtype: dtype = float16
    variant: str = "fp16"
    use_safetensors: bool = True


class Models(Enum):
    FLUX2_DEV = Text2ImageModel(name="black-forest-labs/FLUX.2-dev")
    FLUX2_KLEIN_9B = Text2ImageModel(name="black-forest-labs/FLUX.2-klein-9B")
    FLUX2_KLEIN_4B = Text2ImageModel(name="black-forest-labs/FLUX.2-klein-4B")
    QWEN_IMAGE_2_1 = Text2ImageModel(name="Qwen/Qwen-Image-2.1")
    SD35_LARGE = Text2ImageModel(name="stabilityai/stable-diffusion-3.5-large")
    FLUX1_DEV = Text2ImageModel(name="black-forest-labs/FLUX.1-dev")
    FLUX1_SCHNELL = Text2ImageModel(name="black-forest-labs/FLUX.1-schnell")

In [ ]:
# select text-to-image model
model: Text2ImageModel = Models.FLUX2_DEV.value

In [26]:
from diffusers.pipelines.auto_pipeline import AutoPipelineForText2Image

# define pipeline object
pipe: AutoPipelineForText2Image = AutoPipelineForText2Image.from_pretrained(
    pretrained_model_or_path="stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=float16, variant="fp16", use_safetensors=True
).to("cuda")

/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[transformers] `CLIPImageProcessor` requires torchvision (not installed); falling back to `CLIPImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `CLIPImageProcessorPil` directly to silence this warning.
[transformers] `SiglipImageProcessor` requires torchvision (not installed); falling back to `SiglipImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `SiglipImageProcessorPil` directly to silence this warning.
[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-pac

model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

In [27]:
# wrapper function for generation + persisting to disk
def generate_image(prompt, save_path="app-mockup.png") -> None:
    image = pipe(prompt).images[0]
    image.save(save_path)

In [28]:
generate_image(prompt="A clean UI mockup for a homelab dashboard")

  0%|          | 0/50 [00:00<?, ?it/s]

/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
